In [1]:
# Install required packages.
!pip install \
  langchain==0.1.16 \
  langchain-core==0.1.52 \
  langchain-community==0.0.34 \
  langchain-google-genai==1.0.3 \
  google-search-results \
  requests

In [2]:
import os
import requests
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import SerpAPIWrapper
from langchain_core.tools import Tool
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from google.colab import userdata

In [3]:
# 1. Setup API Keys
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
os.environ["SERPAPI_API_KEY"] = userdata.get('SERPAPI_API_KEY')
OPENWEATHER_API_KEY = userdata.get('OPENWEATHER_API_KEY')

In [4]:
# 2. Define Custom Weather Tool
def get_weather(location: str):
    try:
        url = f"http://api.openweathermap.org/data/2.5/weather?q={location}&appid={OPENWEATHER_API_KEY}&units=metric"
        data = requests.get(url).json()

        if data.get("cod") != 200:
            return f"Could not find weather for {location}"

        desc = data['weather'][0]['description']
        temp = data['main']['temp']
        feels = data['main']['feels_like']

        return f"Weather in {location}: {desc}, Temp: {temp}°C, Feels like: {feels}°C"
    except Exception as e:
        return "Weather service error"

# 3. Initialize Gemini 2.5 Flash
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# 4. Initialize Search Tools
search = SerpAPIWrapper()

tools = [
    Tool(
        name="Weather_Info",
        func=get_weather,
        description="Use this to get current weather and temperature of a city."
    ),
    Tool(
        name="Search",
        func=search.run,
        description=(
            "Use this for travel-related information such as tourist attractions, hotels and itineraries."
        )
    )
]

# 5. Create System Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert travel planner. "
     "Always:\n"
     "- Check weather before suggesting packing\n"
     "- Suggest hotels with brief descriptions and approximate price range (budget, mid-range, luxury)\n"
     "- Create clear day-wise itineraries\n"
     "- Use bullet points and structured format\n"
     "- Use tools when needed for real-time info but STRICTLY use one tool at a time \n"
     ),

    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

# 6. Create the Agent
agent = create_tool_calling_agent(llm, tools, prompt)

# Limit agent looping (important) : max_iterations=5
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=5
)

# 7. Run the Planner
# Ask user to enter city name to prepare travel plan.
city = input("Enter your travel destination: ")


query = f"""
I'm traveling to {city} next week.

Create a complete travel plan including:

1. Weather summary and packing list as per weather condition
2. Top 3 must-see attractions
3. Hotel recommendations (budget and mid-range)
4. A clear 3-day itinerary

Keep it structured and practical.
"""

# Gemini → supports parallel function calling
# LangChain → expects sequential function calling
response = agent_executor.invoke({"input": query})

print(response["output"])

Enter your travel destination: Bangkok


> Entering new AgentExecutor chain...

Invoking: `Weather_Info` with `Bangkok`


Weather in Bangkok: clear sky, Temp: 33.94°C, Feels like: 40.41°C
Invoking: `Search` with `Top 3 must-see attractions in Bangkok`


[{'title': 'Wat Arun Ratchawararam Ratchawaramahawihan', 'description': 'Open', 'rating': 4.7, 'reviews': 44000, 'price': '$6.10', 'extracted_price': 6.1, 'thumbnail': 'https://serpapi.com/searches/69bfc6739f07003a3ec279a7/images/2Tb0N4YszlDUZEVxBvq4zY9IWXLfE4HWmAfG7WZP7Js.jpeg'}, {'title': 'The Grand Palace', 'description': 'Closed', 'rating': 4.6, 'reviews': 77000, 'price': '$15.25', 'extracted_price': 15.25, 'thumbnail': 'https://serpapi.com/searches/69bfc6739f07003a3ec279a7/images/2Tb0N4YszlDUZEVxBvq4zUjmKyLMmM2YLzUvgR2X1tg.jpeg'}, {'title': 'The Temple of the Emerald Buddha', 'description': 'Closed', 'rating': 4.7, 'reviews': 42000, 'price': '$15.25', 'extracted_price': 15.25, 'thumbnail': 'https://serpapi.com/searches/69bfc6739f07

Exception: Multiple function calls are not currently supported